##  Basic Library imports

In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from PIL import Image
import timm
import torch
from tqdm import tqdm

In [ ]:
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/student_resource/'

In [ ]:
DATASET_PATH = os.path.join(DRIVE_PATH, 'dataset/')
IMAGE_PATH = os.path.join(DRIVE_PATH, 'images/')
os.makedirs(IMAGE_PATH, exist_ok=True)
print("Environment setup complete.")

Environment setup complete.


##  Read Dataset

In [ ]:
train_df = pd.read_csv(os.path.join(DATASET_PATH, '/content/drive/MyDrive/student_resource/dataset/train.csv'))
test_df = pd.read_csv(os.path.join(DATASET_PATH, '/content/drive/MyDrive/student_resource/dataset/test.csv'))
# Apply log transformation to the price
train_df['log_price'] = np.log1p(train_df['price'])
# Combine for consistent feature engineering
full_df = pd.concat([train_df.drop('log_price', axis=1), test_df], ignore_index=True)
print(f"Combined data shape: {full_df.shape}")
print("Data loading and log transformation complete.")

Combined data shape: (150000, 4)
Data loading and log transformation complete.


In [ ]:
def create_basic_text_features(df):
    df['content_len'] = df['catalog_content'].str.len()
    df['word_count'] = df['catalog_content'].str.split().str.len()
    df['ipq'] = df['catalog_content'].str.extract(
        r'(\d+\.?\d*)\s*(?:oz|ounce|pack|count|ct|lb|kg|g|fl oz)', flags=re.IGNORECASE
    ).astype(float).fillna(1)
    return df

def create_claim_flags(df):
    content = df['catalog_content'].astype(str).str.lower()
    df['is_gluten_free'] = content.str.contains(r'gluten\W*free|wheat\W*free', flags=re.IGNORECASE).astype(int)
    df['is_non_gmo'] = content.str.contains(r'non\W*gmo|not\W*genetically\W*modified', flags=re.IGNORECASE).astype(int)
    df['is_dairy_free'] = content.str.contains(r'dairy\W*free|non\W*dairy|lactose\W*free', flags=re.IGNORECASE).astype(int)
    return df

def extract_nutrition_features(df):
    content = df['catalog_content'].astype(str).str.lower()
    df['grams'] = content.str.extract(r'(\d+)\s*g', expand=False).astype(float)
    df['protein'] = content.str.extract(r'(\d+)\s*g\s*protein', expand=False).astype(float)
    df['carbs'] = content.str.extract(r'(\d+)\s*g\s*carb', expand=False).astype(float)
    df['sugar'] = content.str.extract(r'(\d+)\s*g\s*sugar', expand=False).astype(float)
    df.fillna({'grams':0,'protein':0,'carbs':0,'sugar':0}, inplace=True)
    return df

def create_certification_flags(df):
    content = df['catalog_content'].astype(str).str.lower()
    df['is_organic_certified'] = content.str.contains(r'usda\s*organic|certified\s*organic').astype(int)
    df['is_fair_trade'] = content.str.contains(r'fair\s*trade').astype(int)
    df['is_iso_certified'] = content.str.contains(r'iso\s*\d+').astype(int)
    df['is_fda_approved'] = content.str.contains(r'fda\s*approved').astype(int)
    return df

def extract_product_category(df):
    content = df['catalog_content'].astype(str).str.lower()
    df['is_snack'] = content.str.contains(r'snack|chips|nuts|bar').astype(int)
    df['is_beverage'] = content.str.contains(r'drink|juice|water|soda|tea|coffee').astype(int)
    df['is_supplement'] = content.str.contains(r'vitamin|protein|supplement').astype(int)
    df['is_bakery'] = content.str.contains(r'bread|cake|cookie|biscuit').astype(int)
    df['is_frozen'] = content.str.contains(r'frozen|ice\s*cream|frosted').astype(int)
    return df

def extract_packaging_features(df):
    content = df['catalog_content'].astype(str).str.lower()
    df['pack_size'] = content.str.extract(r'pack\s*of\s*(\d+)', expand=False).astype(float)
    df['is_eco_friendly'] = content.str.contains(r'eco[-\s]*friendly|recyclable|biodegradable').astype(int)
    df['container_type'] = content.str.extract(r'(bottle|can|pouch|jar|box)', expand=False).fillna('unknown')
    df['container_type'] = LabelEncoder().fit_transform(df['container_type'])
    df.fillna({'pack_size':1}, inplace=True)
    return df

def extract_premium_flags(df):
    content = df['catalog_content'].astype(str).str.lower()
    df['is_premium_word'] = content.str.contains(r'premium|luxury|deluxe|gourmet|handcrafted|exclusive|artisan').astype(int)
    df['is_imported'] = content.str.contains(r'made\s*in|imported\s*from').astype(int)
    return df

def extract_chocolate_features(df):
    content = df['catalog_content'].astype(str).str.lower()
    df['is_chocolate'] = content.str.contains(r'chocolate|cocoa').astype(int)
    df['is_dark_chocolate'] = content.str.contains(r'dark\s*chocolate|70%|85%|cacao').astype(int)
    df['is_milk_chocolate'] = content.str.contains(r'milk\s*chocolate').astype(int)
    df['is_white_chocolate'] = content.str.contains(r'white\s*chocolate').astype(int)
    return df

def encode_brand(df):
    df['brand_name'] = df['catalog_content'].apply(lambda x: str(x).split(':')[0].strip())
    brand_counts = df['brand_name'].value_counts()
    df['brand_encoded'] = df['brand_name'].apply(lambda x: x if brand_counts[x]>5 else 'Rare_Brand')
    le = LabelEncoder()
    df['brand_encoded'] = le.fit_transform(df['brand_encoded'])
    return df

# Apply all feature extraction
full_df = create_basic_text_features(full_df)
full_df = create_claim_flags(full_df)
full_df = extract_nutrition_features(full_df)
full_df = create_certification_flags(full_df)
full_df = extract_product_category(full_df)
full_df = extract_packaging_features(full_df)
full_df = extract_premium_flags(full_df)
full_df = extract_chocolate_features(full_df)
full_df = encode_brand(full_df)

print("✅ All text/categorical features extracted.")

# ------------------------------
# 3. TF-IDF Features
# ------------------------------
tfidf_path = os.path.join(DRIVE_PATH,'tfidf_matrix.npz')
tfidf = TfidfVectorizer(stop_words='english', max_features=10000, dtype=np.float32)

if os.path.exists(tfidf_path):
    print("Loading TF-IDF matrix...")
    full_tfidf_matrix = sp.load_npz(tfidf_path)
else:
    print("Computing TF-IDF matrix...")
    tfidf.fit(full_df['catalog_content'].astype(str))
    full_tfidf_matrix = tfidf.transform(full_df['catalog_content'].astype(str))
    sp.save_npz(tfidf_path, full_tfidf_matrix)
print(f"✅ TF-IDF ready. Shape: {full_tfidf_matrix.shape}")


✅ All text/categorical features extracted.
Computing TF-IDF matrix...
✅ TF-IDF ready. Shape: (150000, 10000)


In [ ]:
semantic_path = os.path.join(DRIVE_PATH,'semantic_embeddings.npy')
if os.path.exists(semantic_path):
    print("Loading semantic embeddings...")
    semantic_embeddings = np.load(semantic_path)
else:
    print("Computing semantic embeddings...")
    model_st = SentenceTransformer('all-MiniLM-L6-v2')
    semantic_embeddings = model_st.encode(full_df['catalog_content'].astype(str).tolist(),
                                          batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    np.save(semantic_path, semantic_embeddings)
print(f"✅ Semantic embeddings ready. Shape: {semantic_embeddings.shape}")

Computing semantic embeddings...


Batches:   0%|          | 0/2344 [00:00<?, ?it/s]

In [ ]:
# ------------------------------
# 5. Image Embeddings
# ------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
img_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0).to(device)
img_model.eval()
data_config = timm.data.resolve_data_config({}, model=img_model)
transforms = timm.data.create_transform(**data_config)

image_path_file = os.path.join(DRIVE_PATH,'image_embeddings.npy')

def get_embedding(image_path):
    try:
        img = Image.open(image_path).convert('RGB')
        img_tensor = transforms(img).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = img_model(img_tensor).cpu().numpy().flatten()
        return emb
    except:
        return np.zeros(1280)

full_df['image_path'] = full_df['image_link'].apply(lambda x: os.path.join(IMAGE_PATH,x.split('/')[-1]))

if os.path.exists(image_path_file):
    print("Loading image embeddings...")
    image_embeddings = np.load(image_path_file)
else:
    print("Computing image embeddings...")
    image_embeddings = np.array([get_embedding(p) for p in tqdm(full_df['image_path'])])
    np.save(image_path_file, image_embeddings)
print(f"✅ Image embeddings ready. Shape: {image_embeddings.shape}")

In [ ]:
feature_cols = [
    'content_len','word_count','ipq','is_gluten_free','is_non_gmo','is_dairy_free',
    'grams','protein','carbs','sugar','is_organic_certified','is_fair_trade','is_iso_certified','is_fda_approved',
    'is_snack','is_beverage','is_supplement','is_bakery','is_frozen',
    'pack_size','is_eco_friendly','container_type','is_premium_word','is_imported',
    'brand_encoded','is_chocolate','is_dark_chocolate','is_milk_chocolate','is_white_chocolate'
]

precomputed_csv = os.path.join(DRIVE_PATH,'precomputed_text_features.csv')
full_df[feature_cols].to_csv(precomputed_csv,index=False)
print(f"✅ Precomputed CSV saved at {precomputed_csv}")

In [ ]:
# ------------------------------
# 7. Split train/test and prepare matrices
# ------------------------------
n_train = len(train_df)

X_train = sp.hstack([
    full_tfidf_matrix[:n_train],
    full_df[feature_cols].iloc[:n_train].values,
    semantic_embeddings[:n_train],
    image_embeddings[:n_train]
]).tocsr()

X_test = sp.hstack([
    full_tfidf_matrix[n_train:],
    full_df[feature_cols].iloc[n_train:].values,
    semantic_embeddings[n_train:],
    image_embeddings[n_train:]
]).tocsr()

y_train = np.log1p(train_df['price'].values)  # log price

print("✅ Model-ready matrices prepared.")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}, y_train shape: {y_train.shape}")


In [ ]:
print("\n--- Part 7: Creating submission file ---")
submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': final_predictions
})

# Verification Step
print(f"Number of rows in submission DataFrame: {len(submission_df)}")

# Save to CSV in the required format
submission_df.to_csv('test_out_updated.csv', index=False)

print("\nSubmission file 'test_out_updated.csv' created successfully!")
print("Here's the head of your submission file:")
print(submission_df.head())